<a href="https://colab.research.google.com/github/Tumu-Moulya-sri/yelp-word-vectorizer-nlp/blob/main/yelp_word_vectorizer_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Final Update - 30 March 2026
!pip install -q datasets transformers torch scikit-learn gensim nltk

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import time
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Load Yelp Review Full dataset
print("Loading Yelp dataset from Hugging Face...")
dataset = load_dataset("Yelp/yelp_review_full", split="train")

# Convert to pandas (faster for sampling)
df = pd.DataFrame(dataset)

# Map labels: 0=1star, 1=2star, ..., 4=5star
df['label'] = df['label'].astype(int)

print(f"Original dataset size: {len(df)}")
print(df['label'].value_counts().sort_index())

In [ ]:
# Stratified sampling: 10,000 reviews per star rating (total 50k)
df_sample = df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=10000, random_state=42))
df_sample = df_sample.reset_index(drop=True)

print(f"Final sample size: {len(df_sample)}")
print(df_sample['label'].value_counts().sort_index())

# Split into train/test (80-20)
train_df, test_df = train_test_split(df_sample, test_size=0.2, stratify=df_sample['label'], random_state=42)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove special characters, numbers, punctuation
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stopwords + lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

# Apply preprocessing (this takes ~2-3 minutes)
print("Preprocessing training data...")
start = time.time()
train_df['clean_text'] = train_df['text'].apply(preprocess_text)
test_df['clean_text'] = test_df['text'].apply(preprocess_text)
print(f"Preprocessing completed in {time.time()-start:.2f} seconds")

In [ ]:
print("=== BoW + Logistic Regression ===")
start = time.time()

vectorizer = CountVectorizer(max_features=20000, ngram_range=(1,2))
X_train = vectorizer.fit_transform(train_df['clean_text'])
X_test = vectorizer.transform(test_df['clean_text'])

model = LogisticRegression(max_iter=500)
model.fit(X_train, train_df['label'])

pred = model.predict(X_test)
acc = accuracy_score(test_df['label'], pred)
print(f"Accuracy: {acc*100:.2f}%")
print(classification_report(test_df['label'], pred, target_names=[f"{i+1}-star" for i in range(5)]))
print(f"Total time: {time.time()-start:.2f} seconds")

In [ ]:
print("=== TF-IDF + Logistic Regression ===")
start = time.time()

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)
X_train = vectorizer.fit_transform(train_df['clean_text'])
X_test = vectorizer.transform(test_df['clean_text'])

model = LogisticRegression(max_iter=500)
model.fit(X_train, train_df['label'])

pred = model.predict(X_test)
acc = accuracy_score(test_df['label'], pred)
print(f"Accuracy: {acc*100:.2f}%")
print(classification_report(test_df['label'], pred, target_names=[f"{i+1}-star" for i in range(5)]))
print(f"Total time: {time.time()-start:.2f} seconds")

In [ ]:
print("=== TF-IDF + Linear SVM ===")
start = time.time()

# Reuse the same TF-IDF vectors from above
model = LinearSVC(max_iter=1000)
model.fit(X_train, train_df['label'])

pred = model.predict(X_test)
acc = accuracy_score(test_df['label'], pred)
print(f"Accuracy: {acc*100:.2f}%")
print(classification_report(test_df['label'], pred, target_names=[f"{i+1}-star" for i in range(5)]))
print(f"Total time: {time.time()-start:.2f} seconds")

In [ ]:
from gensim.models import Word2Vec
import numpy as np

print("=== Training Word2Vec ===")
start = time.time()

# Tokenize for Word2Vec
sentences = [text.split() for text in train_df['clean_text']]

w2v_model = Word2Vec(sentences, vector_size=200, window=5, min_count=3, sg=1, epochs=10, workers=4)

# Convert reviews to vectors (mean pooling)
def get_w2v_vector(text):
    words = text.split()
    vectors = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(200)

X_train_w2v = np.array([get_w2v_vector(text) for text in train_df['clean_text']])
X_test_w2v = np.array([get_w2v_vector(text) for text in test_df['clean_text']])

model = LogisticRegression(max_iter=500)
model.fit(X_train_w2v, train_df['label'])

pred = model.predict(X_test_w2v)
acc = accuracy_score(test_df['label'], pred)
print(f"Accuracy: {acc*100:.2f}%")
print(f"Total time: {time.time()-start:.2f} seconds")

In [ ]:
print("=== GloVe (Pre-trained) ===")
start = time.time()

# Download GloVe 100d
!wget -q https://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

# Load GloVe
glove_dict = {}
with open('glove.6B.100d.txt', 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        glove_dict[word] = vector

def get_glove_vector(text):
    words = text.split()
    vectors = [glove_dict[word] for word in words if word in glove_dict]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

X_train_glove = np.array([get_glove_vector(text) for text in train_df['clean_text']])
X_test_glove = np.array([get_glove_vector(text) for text in test_df['clean_text']])

model = LogisticRegression(max_iter=500)
model.fit(X_train_glove, train_df['label'])

pred = model.predict(X_test_glove)
acc = accuracy_score(test_df['label'], pred)
print(f"Accuracy: {acc*100:.2f}%")
print(f"Total time: {time.time()-start:.2f} seconds")

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

print("=== BERT (Frozen) - Using 5,000 samples ===")
start = time.time()

# Sample 5k for BERT (Colab memory limit)
sample_idx = np.random.choice(len(train_df), 5000, replace=False)
train_small = train_df.iloc[sample_idx]

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model_bert = BertModel.from_pretrained('bert-base-uncased')
model_bert.eval()

def get_bert_embedding(text):
    inputs = tokenizer(text, max_length=64, truncation=True, padding='max_length', return_tensors='pt')
    with torch.no_grad():
        outputs = model_bert(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()  # [CLS] token

X_train_bert = np.array([get_bert_embedding(text) for text in train_small['clean_text']])
X_test_bert = np.array([get_bert_embedding(text) for text in test_df['clean_text'][:2000]])  # smaller test

lr = LogisticRegression(max_iter=200)
lr.fit(X_train_bert, train_small['label'])

pred = lr.predict(X_test_bert[:2000])
acc = accuracy_score(test_df['label'].iloc[:2000], pred)
print(f"Accuracy: {acc*100:.2f}%")
print(f"Total time: {time.time()-start:.2f} seconds")